# Lab 1: Data Visualization, Data Preprocessing, and Statistical Analysis Using Python

**Student Name:** Anna Levinskaia  
**Course:** 2026 Summer – Advanced Big Data and Data Mining (MSCS-634-B01) – Second Bi-term  
**Assignment:** Lab 1: Data Visualization, Data Preprocessing, and Statistical Analysis Using Python in Jupyter Notebook

## Dataset Description

This lab uses a sales dataset containing daily sales transactions. The dataset includes the date, product category, region, units sold, unit price, discount percentage, customer rating, and total sales. The dataset was designed to include missing values and outliers so that data preprocessing techniques can be demonstrated.


## Step 1: Data Collection

In [ ]:
# Import the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display all columns
pd.set_option("display.max_columns", None)

# Load the dataset
df = pd.read_csv("sales_data.csv")

# Convert Date to datetime format
df["Date"] = pd.to_datetime(df["Date"])

# Display the first five rows
df.head()


### Initial Observation

The first five rows show the structure of the dataset. Each row represents one day of sales activity. The dataset contains both categorical variables, such as product category and region, and numerical variables, such as units sold, unit price, customer rating, and sales.


## Step 2: Data Visualization

### Visualization 1: Scatter Plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Units_Sold"], df["Sales"], alpha=0.7)
plt.title("Units Sold vs. Total Sales")
plt.xlabel("Units Sold")
plt.ylabel("Total Sales ($)")
plt.grid(True)
plt.show()


**Insight:** The scatter plot generally shows that total sales increase as the number of units sold increases. A few points are much higher than the rest, suggesting possible outliers in the Sales variable.


### Visualization 2: Line Plot

In [ ]:
daily_sales = df.groupby("Date")["Sales"].sum()

plt.figure(figsize=(10, 5))
plt.plot(daily_sales.index, daily_sales.values)
plt.title("Daily Sales Trend")
plt.xlabel("Date")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


**Insight:** The line plot shows how sales change over time. Most daily sales values remain within a similar range, but several sharp increases indicate unusually high sales records that may need further investigation.


### Visualization 3: Bar Chart

In [ ]:
category_sales = df.groupby("Product_Category")["Sales"].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 5))
category_sales.plot(kind="bar")
plt.title("Total Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Total Sales ($)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


**Insight:** The bar chart compares total sales across product categories. Categories with higher bars contributed more revenue to the dataset.


### Visualization 4: Histogram

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["Sales"].dropna(), bins=20, edgecolor="black")
plt.title("Distribution of Sales")
plt.xlabel("Sales ($)")
plt.ylabel("Frequency")
plt.show()


**Insight:** Most sales values are concentrated in the lower and middle ranges. The long right tail shows that a small number of transactions have unusually high sales values.


### Visualization 5: Box Plot

In [ ]:
plt.figure(figsize=(8, 5))
plt.boxplot(df["Sales"].dropna(), vert=False)
plt.title("Box Plot of Sales")
plt.xlabel("Sales ($)")
plt.show()


**Insight:** The box plot shows the median, quartiles, and possible outliers. The points beyond the whiskers represent unusually high sales values.


### Visualization 6: Pie Chart

In [ ]:
category_counts = df["Product_Category"].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(category_counts.values, labels=category_counts.index, autopct="%1.1f%%", startangle=90)
plt.title("Proportion of Records by Product Category")
plt.show()


**Insight:** The pie chart shows the proportion of records belonging to each product category. This helps determine whether the dataset is balanced across categories.


## Step 3: Data Preprocessing

### 3.1 Handling Missing Values

In [ ]:
# Report missing values before cleaning
print("Missing values before handling:")
print(df.isnull().sum())

# Display rows containing missing values
df[df.isnull().any(axis=1)]


In [ ]:
# Create a copy before preprocessing
df_clean = df.copy()

# Fill numerical missing values with the mean
df_clean["Unit_Price"] = df_clean["Unit_Price"].fillna(df_clean["Unit_Price"].mean())
df_clean["Customer_Rating"] = df_clean["Customer_Rating"].fillna(
    df_clean["Customer_Rating"].mean()
)

# Fill categorical missing values with the mode
df_clean["Region"] = df_clean["Region"].fillna(df_clean["Region"].mode()[0])

print("Missing values after handling:")
print(df_clean.isnull().sum())

# Display the cleaned rows that originally contained missing values
df_clean.loc[df[df.isnull().any(axis=1)].index]


**Decision:** Numerical missing values were replaced with the mean because the mean preserves the overall numerical pattern. Missing Region values were replaced with the mode because Region is categorical.


### 3.2 Outlier Detection and Removal Using IQR

In [ ]:
# Calculate Q1, Q3, and IQR for Sales
Q1 = df_clean["Sales"].quantile(0.25)
Q3 = df_clean["Sales"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Bound:", lower_bound)
print("Upper Bound:", upper_bound)

# Identify outliers
sales_outliers = df_clean[
    (df_clean["Sales"] < lower_bound) |
    (df_clean["Sales"] > upper_bound)
]

print("\nIdentified Sales Outliers:")
sales_outliers


In [ ]:
# Remove Sales outliers
df_no_outliers = df_clean[
    (df_clean["Sales"] >= lower_bound) &
    (df_clean["Sales"] <= upper_bound)
].copy()

print("Dataset shape before outlier removal:", df_clean.shape)
print("Dataset shape after outlier removal:", df_no_outliers.shape)

df_no_outliers.head()


**Decision:** Sales values outside 1.5 times the IQR were removed. This reduces the effect of extreme values on later statistical analysis.


### 3.3 Data Reduction

In [ ]:
print("Dataset before reduction:")
print("Shape:", df_no_outliers.shape)
df_no_outliers.head()


In [ ]:
# Sampling: retain 70% of the cleaned data
df_sampled = df_no_outliers.sample(frac=0.70, random_state=42)

# Dimension reduction: drop a less relevant column
df_reduced = df_sampled.drop(columns=["Discount_Percent"])

print("Dataset after reduction:")
print("Shape:", df_reduced.shape)
df_reduced.head()


**Decision:** A 70% random sample was selected to demonstrate row reduction. The Discount_Percent column was removed to demonstrate dimension elimination.


### 3.4 Data Scaling and Discretization

In [ ]:
# Display numerical columns before scaling
df_reduced[["Units_Sold", "Unit_Price", "Sales"]].head()


In [ ]:
# Min-Max Scaling
df_scaled = df_reduced.copy()

columns_to_scale = ["Units_Sold", "Unit_Price", "Sales"]

for column in columns_to_scale:
    minimum = df_scaled[column].min()
    maximum = df_scaled[column].max()
    df_scaled[column + "_Scaled"] = (
        (df_scaled[column] - minimum) / (maximum - minimum)
    )

# Discretize Sales into three categories
df_scaled["Sales_Level"] = pd.cut(
    df_scaled["Sales"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

df_scaled[
    [
        "Units_Sold",
        "Units_Sold_Scaled",
        "Unit_Price",
        "Unit_Price_Scaled",
        "Sales",
        "Sales_Scaled",
        "Sales_Level"
    ]
].head(10)


**Decision:** Min-Max scaling transformed selected numerical values to a range from 0 to 1. Sales values were also divided into Low, Medium, and High categories to demonstrate discretization.


## Step 4: Statistical Analysis

### 4.1 General Overview of Data

In [ ]:
df_no_outliers.info()


In [ ]:
df_no_outliers.describe(include="all")


### 4.2 Central Tendency Measures

In [ ]:
numeric_columns = df_no_outliers.select_dtypes(include=np.number).columns

central_tendency = pd.DataFrame({
    "Minimum": df_no_outliers[numeric_columns].min(),
    "Maximum": df_no_outliers[numeric_columns].max(),
    "Mean": df_no_outliers[numeric_columns].mean(),
    "Median": df_no_outliers[numeric_columns].median(),
    "Mode": df_no_outliers[numeric_columns].mode().iloc[0]
})

central_tendency


**Interpretation:** The mean and median show the center of each numerical variable. When the mean and median are close, the data is generally balanced. Large differences may indicate skewness or remaining extreme values.


### 4.3 Dispersion Measures

In [ ]:
Q1_all = df_no_outliers[numeric_columns].quantile(0.25)
Q3_all = df_no_outliers[numeric_columns].quantile(0.75)

dispersion = pd.DataFrame({
    "Range": df_no_outliers[numeric_columns].max() - df_no_outliers[numeric_columns].min(),
    "Q1": Q1_all,
    "Q3": Q3_all,
    "IQR": Q3_all - Q1_all,
    "Variance": df_no_outliers[numeric_columns].var(),
    "Standard_Deviation": df_no_outliers[numeric_columns].std()
})

dispersion


**Interpretation:** The range shows the full spread of each variable. The IQR describes the spread of the middle 50% of values. Variance and standard deviation show how far values typically vary from the mean.


### 4.4 Correlation Analysis

In [ ]:
correlation_matrix = df_no_outliers[numeric_columns].corr()
correlation_matrix


In [ ]:
# Optional correlation heatmap using Matplotlib
plt.figure(figsize=(9, 7))
plt.imshow(correlation_matrix, cmap="coolwarm", aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(numeric_columns)), numeric_columns, rotation=45, ha="right")
plt.yticks(range(len(numeric_columns)), numeric_columns)

for i in range(len(numeric_columns)):
    for j in range(len(numeric_columns)):
        plt.text(
            j,
            i,
            f"{correlation_matrix.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

plt.title("Correlation Matrix Heatmap")
plt.tight_layout()
plt.show()


**Interpretation:** Correlation values range from -1 to 1. Values close to 1 indicate a strong positive relationship, values close to -1 indicate a strong negative relationship, and values close to 0 indicate a weak linear relationship. Units Sold and Sales are expected to have a positive relationship.


## Conclusion

This lab demonstrated the complete data analysis process using Python and Jupyter Notebook. The dataset was loaded into a Pandas DataFrame, explored through multiple visualizations, cleaned by handling missing values and outliers, reduced through sampling and column removal, and transformed using scaling and discretization. Statistical measures were then calculated to better understand the center, spread, and relationships between numerical variables.
